# 🚘 CoreVision — Brand-Specific Model Sub-Classifier
Strategy: **frozen backbone + long head training + MixUp** — optimal for small datasets (<100 imgs/class)

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if torch.cuda.is_available(): print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm
print('Done.')

In [ ]:
# ⚙️ CONFIG — only edit this cell per brand
import os, torch

BRAND_NAME   = 'volkswagen'
ZIP_FILENAME = 'VW.zip'

DRIVE_BASE  = '/content/drive/MyDrive/CoreVision'
ZIP_PATH    = os.path.join(DRIVE_BASE, 'data', 'brand_zips', ZIP_FILENAME)
WEIGHTS_OUT = os.path.join(DRIVE_BASE, 'weights', 'brand_model_classifiers')
CLASSES_OUT = os.path.join(DRIVE_BASE, 'data', 'brand_model_classes')
os.makedirs(WEIGHTS_OUT, exist_ok=True)
os.makedirs(CLASSES_OUT, exist_ok=True)

IMG_SIZE   = 224
MAX_BS     = 64
NUM_WORKERS = 4
VAL_RATIO  = 0.15   # 85/15 split — more training images
MIN_IMAGES = 10
EPOCHS     = 60     # head-only, frozen backbone — needs more epochs
LR         = 5e-3   # higher LR is fine when backbone is frozen
WEIGHT_DECAY = 5e-4
GRAD_CLIP  = 2.0
MIXUP_ALPHA = 0.3   # MixUp regularisation — critical for small datasets
SEED       = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Brand: {BRAND_NAME} | Device: {DEVICE}')

In [ ]:
# Extract zip
import zipfile, shutil, time

EXTRACT_ROOT = '/content/extracted'
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f'Not found: {ZIP_PATH}')
if os.path.exists(EXTRACT_ROOT): shutil.rmtree(EXTRACT_ROOT)

t0 = time.time()
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_ROOT)
print(f'Extracted in {time.time()-t0:.0f}s')

top_dirs = [d for d in os.listdir(EXTRACT_ROOT)
            if os.path.isdir(os.path.join(EXTRACT_ROOT,d)) and not d.startswith('.')]
if len(top_dirs) == 1:
    DATA_DIR = os.path.join(EXTRACT_ROOT, top_dirs[0])
else:
    match = [d for d in top_dirs if d.lower()==BRAND_NAME.lower()]
    DATA_DIR = os.path.join(EXTRACT_ROOT, match[0]) if match else EXTRACT_ROOT

IMG_EXTS = ('.jpg','.jpeg','.png','.bmp','.webp')
model_folders = sorted([d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR,d)) and not d.startswith('.')])
print(f'Folder: {DATA_DIR} | Models: {model_folders}')

In [ ]:
# Scan dataset
class_to_paths = {}
for model_name in model_folders:
    folder = os.path.join(DATA_DIR, model_name)
    images = [os.path.join(folder,f) for f in os.listdir(folder) if f.lower().endswith(IMG_EXTS)]
    if len(images) >= MIN_IMAGES:
        class_to_paths[model_name] = sorted(images)
        print(f'  ✅ {model_name}: {len(images)} images')
    else:
        print(f'  ⚠️  {model_name}: {len(images)} (skipped)')

CLASS_NAMES = sorted(class_to_paths.keys())
NUM_CLASSES = len(CLASS_NAMES)
total_imgs  = sum(len(v) for v in class_to_paths.values())
if NUM_CLASSES < 2: raise ValueError(f'Need >= 2 classes, got {NUM_CLASSES}')
print(f'\nClasses ({NUM_CLASSES}): {CLASS_NAMES} | Total: {total_imgs}')

In [ ]:
# Dataset + DataLoaders
import random
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

random.seed(SEED); torch.manual_seed(SEED)

train_samples, val_samples = [], []
for label, cls in enumerate(CLASS_NAMES):
    paths = list(class_to_paths[cls])
    random.shuffle(paths)
    n_val = max(1, min(int(len(paths)*VAL_RATIO), len(paths)-1))
    val_samples   += [(p, label) for p in paths[:n_val]]
    train_samples += [(p, label) for p in paths[n_val:]]

random.shuffle(train_samples); random.shuffle(val_samples)
BATCH_SIZE = min(MAX_BS, len(train_samples))
do_drop    = len(train_samples) > BATCH_SIZE
print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Batch: {BATCH_SIZE}')

class ImageListDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples; self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

# Strong augmentation — essential to regularise with small data
train_tf = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.08),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomRotation(15),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02,0.15)),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

pin = torch.cuda.is_available()
train_loader = DataLoader(ImageListDataset(train_samples, train_tf),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=pin, persistent_workers=True, drop_last=do_drop)
val_loader = DataLoader(ImageListDataset(val_samples, val_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=pin, persistent_workers=True)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
assert len(train_loader) > 0, 'Train loader empty!'

In [ ]:
# Build model — backbone FULLY FROZEN, train head only
# With <100 images/class, updating 5M backbone params causes overfitting.
# Frozen backbone = pretrained ImageNet features + lightweight head = much better generalisation.
import timm
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)

# Freeze backbone completely
for name, p in model.named_parameters():
    p.requires_grad = 'classifier' in name

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
model = model.to(DEVICE)
print(f'EfficientNet-B0 | trainable: {trainable:,} / {total:,} params ({trainable/total:.1%})')
print(f'Classes: {CLASS_NAMES}')

In [ ]:
# Training — frozen backbone, MixUp, ReduceLROnPlateau
import torch.nn as nn, torch.optim as optim, time, numpy as np

amp_enabled = DEVICE.type == 'cuda'
scaler      = torch.amp.GradScaler('cuda', enabled=amp_enabled)

def mixup_batch(imgs, labels, alpha=0.3):
    """MixUp: blend pairs of images and their labels."""
    if alpha <= 0: return imgs, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    return mixed, labels, labels[idx], lam

def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=8, min_lr=1e-6, verbose=True)

def run_train_epoch(model, loader):
    model.train()
    total_loss, correct, count = 0.0, 0, 0
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        imgs, y_a, y_b, lam = mixup_batch(imgs, labels, MIXUP_ALPHA)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=amp_enabled):
            logits = model(imgs)
            loss   = mixup_loss(criterion, logits, y_a, y_b, lam)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * labels.size(0)
        # Accuracy: use original labels (pre-mix) for reporting
        correct += (logits.argmax(1) == y_a).sum().item()
        count   += labels.size(0)
    return total_loss/max(count,1), correct/max(count,1)

def run_val_epoch(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=amp_enabled):
                logits = model(imgs)
                loss   = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            correct    += (logits.argmax(1) == labels).sum().item()
            count      += labels.size(0)
    return total_loss/max(count,1), correct/max(count,1)

best_val_acc, best_state = 0.0, None
print(f'Training for {EPOCHS} epochs | MixUp alpha={MIXUP_ALPHA} | LR={LR}')
print(f'ReduceLROnPlateau: halves LR if val_acc does not improve for 8 epochs\n')

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc = run_train_epoch(model, train_loader)
    vl_loss, vl_acc = run_val_epoch(model, val_loader)
    scheduler.step(vl_acc)
    flag = '✅' if vl_acc > best_val_acc else '  '
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_state   = {k: v.cpu().clone() for k,v in model.state_dict().items()}
    print(f'{flag} [{epoch:>2}/{EPOCHS}]  '
          f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f}  '
          f'vl_loss={vl_loss:.4f} vl_acc={vl_acc:.3f}  '
          f'lr={optimizer.param_groups[0]["lr"]:.2e}  ({time.time()-t0:.0f}s)')

print(f'\n🏆 Best val acc: {best_val_acc:.3f} ({best_val_acc:.1%})')

In [ ]:
# Evaluate with TTA (Test-Time Augmentation) — averages 5 augmented views per image
tta_tf = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
tta_dataset = ImageListDataset(val_samples, tta_tf)

model.load_state_dict(best_state)
model.eval()

N_TTA = 5
all_probs  = [[] for _ in range(len(val_samples))]
all_labels = [s[1] for s in val_samples]

for _ in range(N_TTA):
    tta_loader = DataLoader(tta_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=pin)
    idx = 0
    with torch.no_grad():
        for imgs, _ in tta_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=amp_enabled):
                probs = torch.softmax(model(imgs), dim=1).cpu()
            for p in probs:
                all_probs[idx].append(p)
                idx += 1

final_preds = [torch.stack(ps).mean(0).argmax().item() for ps in all_probs]

print('=== Per-class Val Accuracy (with TTA x5) ===')
for i, cls in enumerate(CLASS_NAMES):
    idx   = [j for j,l in enumerate(all_labels) if l==i]
    right = sum(1 for j in idx if final_preds[j]==i)
    print(f'  {cls}: {right}/{len(idx)} = {right/max(len(idx),1):.1%}')
overall = sum(p==l for p,l in zip(final_preds,all_labels))/max(len(all_labels),1)
print(f'\nOverall (TTA): {overall:.1%}')

In [ ]:
# Save to Drive
from datetime import datetime

weights_path = os.path.join(WEIGHTS_OUT, f'{BRAND_NAME}_model_classifier.pth')
classes_path = os.path.join(CLASSES_OUT, f'{BRAND_NAME}_model_classes.txt')

torch.save({
    'model_state_dict': best_state,
    'class_names':      CLASS_NAMES,
    'num_classes':      NUM_CLASSES,
    'brand':            BRAND_NAME,
    'backbone':         'efficientnet_b0',
    'best_val_acc':     round(float(best_val_acc), 4),
    'input_size':       IMG_SIZE,
    'frozen_backbone':  True,
    'trained_at_utc':   datetime.utcnow().isoformat() + 'Z',
}, weights_path)

with open(classes_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(CLASS_NAMES))

print(f'✅ {weights_path}')
print(f'✅ {classes_path}')
print(f'   Brand: {BRAND_NAME} | Classes: {CLASS_NAMES} | Val acc: {best_val_acc:.1%}')
print('\nNext brand: change BRAND_NAME + ZIP_FILENAME in Cell 2 and Run All.')